# 04 · Streaming Data Processing

This notebook documents the streaming pipeline that ingests live Wikimedia change events via Kafka and processes them with Spark Structured Streaming.

- **Producer**: `src/streaming_producer.py` (Wikimedia EventStreams -> Kafka).
- **Consumer**: `src/streaming_pipeline.py` (Kafka -> Parquet, with watermark).
- **Topic**: `wiki-changes`.
- **Output**: `data/streaming/output/wiki_changes/`.

## Methodology

We ingest the live Wikimedia EventStreams feed through a Python producer thatfilters changes mentioning movie-related keywords (`film`, `movie`, `actor`, `actress`, `cinema`, `director`) and publishes them to the Kafka topic `wiki-changes`.

Spark Structured Streaming consumes this topic, parses the JSON payload, applies a **10-minute watermark** on the event timestamp to tolerate late arriving data, and writes the enriched events to Parquet with checkpointing.

The filter is intentionally broad to guarantee that several matching events are captured within a 15 to 30 minute test window, as required by the project.

In [15]:
import subprocess

result = subprocess.run(["docker", "ps"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

CONTAINER ID   IMAGE                COMMAND                  CREATED          STATUS          PORTS                                         NAMES
58458f3fc92d   apache/kafka:3.7.0   "/__cacert_entrypoinâ€¦"   49 minutes ago   Up 49 minutes   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp   kafka




## Running the pipeline

Two processes must run in **separate terminals** while this notebook
inspects the output:

1. **Producer** (terminal 1):
   ```bash
   python -m src.streaming_producer
   ```
2. **Spark Pipeline** (terminal 2):
   ```bash
   python -m src.streaming_pipeline
   ```

In [16]:
import os

from pyspark.sql import SparkSession

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
output = os.path.join(project_root, "data", "streaming", "output", "wiki_changes")

print("Reading from:", output)

spark = (
    SparkSession.builder
    .appName("StreamingInspect")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

if os.path.isdir(output):
    df = spark.read.parquet(output + "/*.parquet")
    print(f"Total rows: {df.count():,}")
    df.select("title", "type", "event_time") \
      .orderBy("event_time", ascending=False) \
      .show(10, truncate=False)
else:
    print("No output yet. Start the producer and the pipeline in two terminals.")

Reading from: c:\Users\anasb\BigData\Project\data\streaming\output\wiki_changes
Total rows: 178
+---------------------------------------------------------+----------+-------------------+
|title                                                    |type      |event_time         |
+---------------------------------------------------------+----------+-------------------+
|Q21775771                                                |edit      |2026-11-09 21:10:39|
|Q6978549                                                 |edit      |2026-11-09 21:10:34|
|Talk:List of accolades received by Inside Out (2015 film)|log       |2026-11-09 21:10:33|
|Category:Redirects from moves                            |categorize|2026-11-09 21:10:33|
|List of accolades received by Inside Out (2015 film)     |log       |2026-11-09 21:10:32|
|Talk:List of accolades received by Inside Out            |log       |2026-11-09 21:10:32|
|Category:Redirects from moves                            |categorize|2026-11-09 21:1

## Late data handling

Spark Structured Streaming uses **event-time processing** with a `withWatermark("event_time", "10 minutes")` on the event timestamp extracted from each Wikimedia change.

- **Why a watermark?** The Wikimedia feed is distributed and events can arrive out of order. Without a watermark, Spark must keep all state indefinitely, which leads to unbounded memory usage. The watermark defines a bounded waiting window: events older than 10 minutes relative to the current maximum event time are considered late and dropped from stateful aggregations.
- **Why 10 minutes?** A short window keeps memory low and output fresh, but risks dropping legitimately delayed events. A long window (1 hour, 2 hour...) tolerates more disorder but keeps more state alive. 10 minutes is a reasonable compromise for a live demo where the producer forwards events within seconds.
- **Output mode**: `append` is used, which is the required mode when a watermark is present and results are finalized batches.

**Alternatives considered**:
- No watermark, relying on `processingTime` only. Simpler but not event-time aware, and incompatible with stateful operations.
- `complete` output mode. Rewrites the full result at each batch, which iswasteful for an append-only event log.
- A longer watermark (30–60 minutes). Higher latency and more state, unjustified for a 15–30 minute test window.

In [17]:
if os.path.isdir(output):
    df2 = spark.read.parquet(output + "/*.parquet")
    print("Events by wiki and change type:")
    df2.groupBy("wiki", "type") \
      .count() \
      .orderBy("count", ascending=False) \
      .show(20, truncate=False)
else:
    print("No output yet.")

Events by wiki and change type:
+------------+----------+-----+
|wiki        |type      |count|
+------------+----------+-----+
|enwiki      |edit      |68   |
|wikidatawiki|edit      |50   |
|commonswiki |categorize|17   |
|enwiki      |categorize|14   |
|enwiki      |log       |11   |
|frwiki      |edit      |6    |
|itwiki      |edit      |5    |
|commonswiki |edit      |5    |
|dewiki      |edit      |4    |
|enwiki      |new       |3    |
|trwiki      |categorize|2    |
|eswiki      |edit      |2    |
|itwiki      |categorize|2    |
|cywiki      |categorize|2    |
|wikidatawiki|new       |2    |
|eswiki      |new       |1    |
|cywiki      |log       |1    |
|trwiki      |new       |1    |
|commonswiki |log       |1    |
|frwiki      |categorize|1    |
+------------+----------+-----+
only showing top 20 rows

